<div style="
background:linear-gradient(135deg,#171922,#10131b);
border:2px solid #3ce7e7;
border-radius:22px;
padding:32px;
font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;
color:white;
box-shadow:0 0 18px rgba(60,231,231,.18);
">

<div style="display:flex;align-items:center;gap:14px;">
<span style="font-size:34px;">👨‍💻</span>

<div>
<div style="
font-size:34px;
font-weight:800;
color:#4ff5f5;
text-shadow:0 0 8px rgba(79,245,245,.35);
">
Aliakbar Isapour
</div>

<div style="
color:#bfbfbf;
font-size:16px;
margin-top:3px;
">
Machine Learning • Data Science • AI
</div>
</div>
</div>

<div style="
height:2px;
background:linear-gradient(90deg,#7b3ff2,#4ff5f5);
border-radius:5px;
margin:22px 0;
"></div>

<div style="
color:#d7d7d7;
font-size:16px;
line-height:1.7;
margin-bottom:25px;
">

</div>


<div style="display:flex;gap:14px;flex-wrap:wrap;">

<a href="https://github.com/isapour0"
target="_blank"
rel="noopener noreferrer"
style="
display:inline-block;
margin: 0 10px;
text-decoration:none;
background:#23283a;
color:#4ff5f5;
padding:12px 22px;
border-radius:12px;
border:1px solid #4ff5f5;
font-weight:bold;
transition:0.25s;
">
GitHub
</a>

<a href="https://www.linkedin.com/in/aliakbar-isapour-682592413"
target="_blank"
rel="noopener noreferrer"
style="
display:inline-block;
margin: 0 10px;
text-decoration:none;
background:#23283a;
color:#4ff5f5;
padding:12px 22px;
border-radius:12px;
border:1px solid #4ff5f5;
font-weight:bold;
transition:0.25s;
">
LinkedIn
</a>


</div>

</div>

<div style="
background:linear-gradient(135deg,#08131f,#0d1b2a,#132238);
border:1px solid #00d4ff;
border-left: 8px solid #00E5FF;
border-radius:18px;
padding:25px;
box-shadow:0 0 25px rgba(0,212,255,.25);
color:white;
font-family:Segoe UI;
">

<h2 style="color:#00E5FF;">📌 Library Imports</h2>
<hr style="border:1px solid #8A2BE2; opacity:.4;">
</div>


In [2]:
# ---- core ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
import warnings

# ---- modelSelection and metrics ----
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# ---- keras ----
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)

classes_name = [
    'Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
    'Dog', 'Frog', 'Horse', 'Ship', 'Truck'
]

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


<div style="
background:linear-gradient(135deg,#08131f,#0d1b2a,#132238);
border:1px solid #00d4ff;
border-radius:18px;
border-left: 8px solid #00E5FF;
padding:25px;
box-shadow:0 0 25px rgba(0,212,255,.25);
color:white;
font-family:Segoe UI;
">
    
<h2 style="color:#00E5FF;">📂 Data Loding</h2>
<hr style="border:1px solid #8A2BE2;opacity:.4;">
</div>


In [3]:
DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

paths = {name: DATA_DIR / f"{name}.npy" for name in ["X_train", "y_train", "X_test", "y_test"]}

if all(p.exists() for p in paths.values()):
    print("Loading CIFAR-10 from local .npy files...")
    X_train = np.load(paths["X_train"])
    y_train = np.load(paths["y_train"])
    X_test = np.load(paths["X_test"])
    y_test = np.load(paths["y_test"])
else:
    print("Downloading CIFAR-10 with Keras...")
    (X_train, y_train), (X_test, y_test) = cifar10.load_data()
    np.save(paths["X_train"], X_train)
    np.save(paths["y_train"], y_train)
    np.save(paths["X_test"], X_test)
    np.save(paths["y_test"], y_test)
    print("Saved to", DATA_DIR.resolve())

Loading CIFAR-10 from local .npy files...


<div style="
background:linear-gradient(135deg,#08131f,#0d1b2a,#132238);
border:1px solid #00d4ff;
border-left: 8px solid #00E5FF;
border-radius:18px;
padding:25px;
box-shadow:0 0 25px rgba(0,212,255,.25);
color:white;
font-family:Segoe UI;
">

<h2 style="color:#00E5FF;">🔧 Preprocessing</h2>
<hr style="border:1px solid #8A2BE2;opacity:.4;">

<ul>
    
<li>Step 1: Separating a 10% validation set.</li> 
<li>Step 2: Scaling pixels to the range [0, 1].</li>
<li>One-hot encoding of the labels (the integer labels stay available for evaluation)</li> 

</ul>
</div>

In [4]:
# Step 1: Separating a 10% validation set
X_train_part, X_val, y_train_part, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    stratify=y_train.ravel(),
    random_state=RANDOM_STATE,
)

print("Train part:", X_train_part.shape, "| Validation:", X_val.shape, "| Test:", X_test.shape)

Train part: (45000, 32, 32, 3) | Validation: (5000, 32, 32, 3) | Test: (10000, 32, 32, 3)


In [5]:
# Step 2: Scaling pixels to the range [0, 1]
def prepare_images(X):
    """uint8 images (N, 32, 32, 3) -> float32 in [0, 1], same shape."""
    if X.dtype != np.uint8:
        raise ValueError(f"Expected uint8 images, got {X.dtype}. Was this data already scaled?")
    return X.astype("float32") / 255.0

X_train_scaled = prepare_images(X_train_part)
X_val_scaled = prepare_images(X_val)
X_test_scaled = prepare_images(X_test)

for name, arr in [("X_train_scaled", X_train_scaled), ("X_val_scaled", X_val_scaled), ("X_test_scaled", X_test_scaled)]:
    print(f"{name:15s} shape={arr.shape}  dtype={arr.dtype}  min={arr.min():.1f}  max={arr.max():.1f}")

X_train_scaled  shape=(45000, 32, 32, 3)  dtype=float32  min=0.0  max=1.0
X_val_scaled    shape=(5000, 32, 32, 3)  dtype=float32  min=0.0  max=1.0
X_test_scaled   shape=(10000, 32, 32, 3)  dtype=float32  min=0.0  max=1.0


In [6]:
# Step 3: One-hot encoding of the labels (the integer labels stay available for evaluation)
y_train_cat = to_categorical(y_train_part, num_classes=10)
y_val_cat = to_categorical(y_val, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print("y_train_cat:", y_train_cat.shape, "| y_val_cat:", y_val_cat.shape, "| y_test_cat:", y_test_cat.shape)

y_train_cat: (45000, 10) | y_val_cat: (5000, 10) | y_test_cat: (10000, 10)


<div style="
background:linear-gradient(135deg,#08131f,#0d1b2a,#132238);
border:1px solid #00d4ff;
border-left: 8px solid #00E5FF;
border-radius:18px;
padding:25px;
box-shadow:0 0 25px rgba(0,212,255,.25);
color:white;
font-family:Segoe UI;
">

<h2 style="color:#00E5FF;">🤖 Baseline CNN Model</h2>
<hr style="border:1px solid #8A2BE2;opacity:.4;">

<li>Step 1: Architecture: 3 x [Conv2D (3x3, ReLU, same padding) + MaxPooling 2x2] with 64, 128 and 128 filters, then Flatten, Dense(128, ReLU) and a softmax output layer with 10 units.</li>
<li>Step 2: Settings: Adam (lr=0.001), batch size 64, 30 epochs, categorical cross-entropy. No dropout, no early stopping, no data augmentation, so the tuning stages later show clearly what each change adds.</li>
<li>Step 3: The model is trained on the 45,000 training images and monitored on the fixed 5,000-image validation set</li>
</div>

In [13]:
model_baseline = Sequential([
    # Input shape data
    Input(shape=(32, 32, 3)),

    # ---- convolutional layers ----
    Conv2D(32, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),

    # ---- Fully Connected ----
    Flatten(),
    Dense(128, activation="relu"),
    Dense(10, activation="softmax"),
])

model_baseline.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 356,810 (1.36 MB)

 Trainable params: 356,810 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model_baseline.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_baseline = model_baseline.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=30,
    batch_size=64,
    verbose=1,
)

model_baseline.save("../models/cnn_baseline.keras")

Epoch 1/30


I0000 00:00:1795697789.054258   77389 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_478780__.43


695/704 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4576 - loss: 1.4913

I0000 00:00:1795697794.384083   77389 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_478780__.43


704/704 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.4592 - loss: 1.4872 - val_accuracy: 0.5826 - val_loss: 1.1797
Epoch 2/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6299 - loss: 1.0509 - val_accuracy: 0.6488 - val_loss: 0.9908
Epoch 3/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6928 - loss: 0.8820 - val_accuracy: 0.6942 - val_loss: 0.8914
Epoch 4/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7322 - loss: 0.7697 - val_accuracy: 0.6932 - val_loss: 0.8727
Epoch 5/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7653 - loss: 0.6798 - val_accuracy: 0.7062 - val_loss: 0.8525
Epoch 6/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7892 - loss: 0.6072 - val_accuracy: 0.7130 - val_loss: 0.8453
Epoch 7/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8104 - loss: 0.5501 - val_accuracy: 0.7220 - val_loss: 0.8365
Epoch 8/30
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8251 - loss: 0.4988 - val_accuracy: 0.7190 - val_

In [ ]:
hist = pd.DataFrame(history_baseline.history)
best_epoch = hist["val_loss"].idxmin() + 1

print("Epoch with the lowest validation loss:", best_epoch)
print("Lowest validation loss:", round(hist["val_loss"].min(), 4))
print("Last epoch -> train acc:", round(hist["accuracy"].iloc[-1], 4),
      "| val acc:", round(hist["val_accuracy"].iloc[-1], 4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist["loss"], label="Training loss")
axes[0].plot(hist["val_loss"], label="Validation loss")
axes[0].axvline(best_epoch - 1, color="gray", linestyle="--", label=f"Lowest val loss (epoch {best_epoch})")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(hist["accuracy"], label="Training accuracy")
axes[1].plot(hist["val_accuracy"], label="Validation accuracy")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
# plt.savefig(FIG_DIR / "training_curves_cnn_baseline.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Accuracy and Error rate analysis
test_loss, test_accuracy = model_baseline.evaluate(X_test_scaled, y_test_cat, verbose=1)

print (f'test loss: {test_loss:.4f}')
print (f'test accuracy: {test_accuracy:.4f}')